# Лаб4.

Егорова Варвара Александровна. P3323, 408575

1. Реализовать класс DecisionTree.
В классе **должны** быть методы
```python
def __init__(..., classification:bool=true, ...):
    # some code
def predict(...):
    # some code
def fit(...):
    # some code
```
2. Реализовать класс RandomForest.
В классе **должны** быть методы
```python
def __init__(..., classification:bool=true, ...):
    # some code
def predict(...):
    # some code
def fit(...):
    # some code
```
3. Реализовать класс GradientBoosting.
В классе **должны** быть методы
```python
def __init__(..., classification:bool=true, ...):
    # some code
def predict(...):
    # some code
def fit(...):
    # some code
```
4. Обучить модели каждого алгоритма на следующих датасетах (проводим мини-соревнование между ними):
    * регрессия: https://www.kaggle.com/datasets/hmavrodiev/london-bike-sharing-dataset
    * классификация: https://www.kaggle.com/datasets/pankrzysiu/cifar10-python

5. Объяснить, почему алгоритм $A$ "победил", а почему алгоритм $B$ "проиграл".

# Пишем класс DescisionTree

In [ ]:
import torch

class DecisionTree:
    def __init__(self, max_depth=10, min_samples_split=2, classification=True):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.classification = classification
        self.tree = None

    def _gini(self, y):
        _, counts = torch.unique(y, return_counts=True)
        p = counts.float() / len(y)
        return 1 - torch.sum(p*p)

    def _variance(self, y):
        return torch.var(y.float())

    def _best_split(self, X, y):
        best_feat, best_thr = None, None
        best_gain = -1e18

        m, n = X.shape

        for feat in range(n):
            col = X[:, feat]
            min_v = torch.min(col)
            max_v = torch.max(col)
            if min_v == max_v:
                continue
            thresholds = torch.linspace(min_v, max_v, steps=16)

            for thr in thresholds:
                left_mask = col <= thr
                right_mask = ~left_mask
                if left_mask.sum() == 0 or right_mask.sum() == 0:
                    continue

                left_y = y[left_mask]
                right_y = y[right_mask]

                if self.classification:
                    impurity_parent = self._gini(y)
                    gain = impurity_parent - (
                        len(left_y)/len(y) * self._gini(left_y) +
                        len(right_y)/len(y) * self._gini(right_y)
                    )
                else:
                    var_parent = self._variance(y)
                    gain = var_parent - (
                        len(left_y)/len(y) * self._variance(left_y) +
                        len(right_y)/len(y) * self._variance(right_y)
                    )

                if gain > best_gain:
                    best_gain = gain
                    best_feat = feat
                    best_thr = float(thr)

        return best_feat, best_thr

    def _build(self, X, y, depth):
        if depth >= self.max_depth or len(y) < self.min_samples_split:
            if self.classification:
                classes, counts = torch.unique(y, return_counts=True)
                return int(classes[torch.argmax(counts)])
            else:
                return float(torch.mean(y.float()))

        feat, thr = self._best_split(X, y)
        if feat is None:
            if self.classification:
                classes, counts = torch.unique(y, return_counts=True)
                return int(classes[torch.argmax(counts)])
            else:
                return float(torch.mean(y.float()))

        left_mask = X[:, feat] <= thr
        right_mask = ~left_mask

        return {
            "feature": feat,
            "threshold": thr,
            "left": self._build(X[left_mask], y[left_mask], depth + 1),
            "right": self._build(X[right_mask], y[right_mask], depth + 1),
        }

    def fit(self, X, y):
        self.X = torch.as_tensor(X, dtype=torch.float32)
        y = torch.as_tensor(y, dtype=torch.int64 if self.classification else torch.float32)
        self.tree = self._build(self.X, y, 0)

    def _predict_one(self, node, x):
        if not isinstance(node, dict):
            return node
        if x[node["feature"]] <= node["threshold"]:
            return self._predict_one(node["left"], x)
        else:
            return self._predict_one(node["right"], x)

    def predict(self, X):
        X = torch.as_tensor(X, dtype=torch.float32)
        return torch.tensor([self._predict_one(self.tree, x) for x in X])


# Пишем класс RandomForest

In [ ]:
import torch
import random

class RandomForest:
    def __init__(self, n_estimators=10, max_depth=10, min_samples_split=2,
                 max_features=None, classification=True):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.max_features = max_features
        self.classification = classification
        self.trees = []

    def _bootstrap(self, X, y):
        n = len(X)
        idxs = torch.randint(0, n, (n,))
        return X[idxs], y[idxs]

    def _subset_features(self, X):
        n_features = X.shape[1]
        if self.max_features is None:
            k = int(n_features ** 0.5) if self.classification else int(n_features * 0.7)
        else:
            k = self.max_features
        idxs = torch.randperm(n_features)[:k]
        return idxs

    def fit(self, X, y):
        X = torch.tensor(X, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.int64 if self.classification else torch.float32)

        for i in range(self.n_estimators):
            print(f"  Tree {i+1}/{self.n_estimators}")
            Xs, ys = self._bootstrap(X, y)
            feat_idx = self._subset_features(X)
            Xt = Xs[:, feat_idx]

            tree = DecisionTree(
                max_depth=self.max_depth,
                min_samples_split=self.min_samples_split,
                classification=self.classification
            )
            tree.fit(Xt, ys)
            self.trees.append((tree, feat_idx))

    def predict(self, X):
        X = torch.tensor(X, dtype=torch.float32)
        preds = []

        for tree, feat_idx in self.trees:
            Xt = X[:, feat_idx]
            preds.append(tree.predict(Xt))

        preds = torch.stack(preds, dim=0)

        if self.classification:
            out = []
            for i in range(preds.shape[1]):
                values, counts = torch.unique(preds[:, i], return_counts=True)
                out.append(int(values[torch.argmax(counts)]))
            return torch.tensor(out)
        else:
            return torch.mean(preds.float(), dim=0)

# Ну и теперь GradientBoosting

In [ ]:
import torch

class GradientBoosting:
    def __init__(self, n_estimators=20, learning_rate=0.1, max_depth=3, classification=True):
        self.n_estimators = n_estimators
        self.learning_rate = learning_rate
        self.max_depth = max_depth
        self.classification = classification
        self.models = []
        self.base = None
        self.num_classes = None
        self.device = torch.device("cpu")

    def fit(self, X, y):
        X = torch.as_tensor(X, dtype=torch.float32)
        self.device = X.device

        if self.classification:
            y = torch.as_tensor(y, dtype=torch.long, device=self.device)
            self.num_classes = int(y.max().item() + 1)

            counts = torch.bincount(y, minlength=self.num_classes).float()
            p = counts / counts.sum()
            self.base = torch.log(p + 1e-9).to(self.device)

            pred = self.base.unsqueeze(0).repeat(len(X), 1)

            for i in range(self.n_estimators):
                print(f"  Estimator {i+1}/{self.n_estimators}")
                prob = torch.softmax(pred, dim=1)
                y_one_hot = torch.nn.functional.one_hot(y, num_classes=self.num_classes).float()

                residual = (y_one_hot - prob).to(self.device)
                trees = []

                for c in range(self.num_classes):
                    t = DecisionTree(max_depth=self.max_depth,
                                     min_samples_split=2,
                                     classification=False)
                    t.fit(X.cpu(), residual[:, c].cpu())
                    update = t.predict(X.cpu()).to(self.device)

                    pred[:, c] += self.learning_rate * update
                    trees.append(t)

                self.models.append(trees)

        else:
            y = torch.as_tensor(y, dtype=torch.float32, device=self.device)
            self.base = y.mean().to(self.device)
            pred = torch.full_like(y, self.base)

            for i in range(self.n_estimators):
                print(f"  Tree {i+1}/{self.n_estimators}")
                residual = (y - pred).to(self.device)

                t = DecisionTree(max_depth=self.max_depth,
                                 min_samples_split=2,
                                 classification=False)

                t.fit(X.cpu(), residual.cpu())
                update = t.predict(X.cpu()).to(self.device)

                pred += self.learning_rate * update
                self.models.append(t)

    def predict(self, X):
        X = torch.as_tensor(X, dtype=torch.float32)
        X_cpu = X.cpu()

        if self.classification:
            pred = self.base.unsqueeze(0).repeat(len(X), 1).to(self.device)

            for trees in self.models:
                for c, t in enumerate(trees):
                    pred[:, c] += self.learning_rate * t.predict(X_cpu).to(self.device)

            return torch.argmax(pred, dim=1)

        else:
            pred = torch.full((len(X),), self.base, device=self.device)

            for t in self.models:
                pred += self.learning_rate * t.predict(X_cpu).to(self.device)

            return pred


# Мои любимые 3 класса

In [ ]:
import numpy as np
from dataclasses import dataclass
import torch

device = torch.device("cpu")

@dataclass
class SeparateData:
  X_train: torch.Tensor
  y_train: torch.Tensor
  X_val: torch.Tensor
  y_val: torch.Tensor
  X_test: torch.Tensor
  y_test: torch.Tensor

class StandardScaler:
    def __init__(self):
        self.mean = None
        self.std = None

    def fit(self, X: torch.Tensor) -> None:
        X = X.clone().detach().to(torch.float32)
        if X.ndim == 1:
            X = X.reshape(-1, 1)

        self.mean = torch.mean(X, dim=0)
        self.std = torch.std(X, dim=0)

        self.std = torch.where(self.std == 0, torch.tensor(1.0), self.std)

    def transform(self, X: torch.Tensor) -> torch.Tensor:
        if self.mean is None or self.std is None:
            return X

        X = X.clone().detach().to(torch.float32)
        if X.ndim == 1:
            X = X.reshape(-1, 1)

        return (X - self.mean) / self.std

    def fit_transform(self, X: torch.Tensor) -> torch.Tensor:
        self.fit(X)
        return self.transform(X)

class DataSeparator:
   def __init__(self, val_size: float = 0.2, test_size: float = 0.1,
                 shuffle: bool = True, seed: int = 42, device: torch.device = device):
        self.val_size = val_size
        self.test_size = test_size
        self.shuffle = shuffle
        self.seed = seed
        self.device = device

   def separate(self, X: torch.tensor, y: torch.tensor) -> SeparateData:
    n_samples = X.shape[0]
    indices = torch.arange(n_samples)

    if self.shuffle:
         torch.manual_seed(self.seed)
         indices = torch.randperm(n_samples)

    test_samples = int(n_samples * self.test_size)
    val_samples = int(n_samples * self.val_size)
    train_samples = n_samples - test_samples - val_samples

    train_indices = indices[:train_samples]
    val_indices = indices[train_samples: train_samples + val_samples]
    test_indices = indices[train_samples + val_samples:]

    X_train = X[train_indices].to(self.device)
    y_train = y[train_indices].to(self.device)
    X_val   = X[val_indices].to(self.device)
    y_val   = y[val_indices].to(self.device)
    X_test  = X[test_indices].to(self.device)
    y_test  = y[test_indices].to(self.device)

    return SeparateData(X_train, y_train, X_val, y_val, X_test, y_test)

# Решаем задачу регрессии

## Работаем с датасетиком

In [ ]:
import pandas as pd
import kagglehub

path = kagglehub.dataset_download("hmavrodiev/london-bike-sharing-dataset")

print("Path to dataset files:", path)

file_name = "london_merged.csv"
full_path = f"{path}/{file_name}"

df = pd.read_csv(full_path)

print("\nПервые 5 записей:")
print(df.head())

Using Colab cache for faster access to the 'london-bike-sharing-dataset' dataset.
Path to dataset files: /kaggle/input/london-bike-sharing-dataset

Первые 5 записей:
             timestamp  cnt   t1   t2    hum  wind_speed  weather_code  \
0  2015-01-04 00:00:00  182  3.0  2.0   93.0         6.0           3.0   
1  2015-01-04 01:00:00  138  3.0  2.5   93.0         5.0           1.0   
2  2015-01-04 02:00:00  134  2.5  2.5   96.5         0.0           1.0   
3  2015-01-04 03:00:00   72  2.0  2.0  100.0         0.0           1.0   
4  2015-01-04 04:00:00   47  2.0  0.0   93.0         6.5           1.0   

   is_holiday  is_weekend  season  
0         0.0         1.0     3.0  
1         0.0         1.0     3.0  
2         0.0         1.0     3.0  
3         0.0         1.0     3.0  
4         0.0         1.0     3.0  


In [ ]:
import pandas as pd
import numpy as np

df['timestamp'] = pd.to_datetime(df['timestamp'])

df['hour'] = df['timestamp'].dt.hour
df['dayofweek'] = df['timestamp'].dt.dayofweek
df['month'] = df['timestamp'].dt.month
df['year'] = df['timestamp'].dt.year

df = df.drop('timestamp', axis=1)

print("Первые 5 записей:")
print(df.head())
print(df.info())

Первые 5 записей:
   cnt   t1   t2    hum  wind_speed  weather_code  is_holiday  is_weekend  \
0  182  3.0  2.0   93.0         6.0           3.0         0.0         1.0   
1  138  3.0  2.5   93.0         5.0           1.0         0.0         1.0   
2  134  2.5  2.5   96.5         0.0           1.0         0.0         1.0   
3   72  2.0  2.0  100.0         0.0           1.0         0.0         1.0   
4   47  2.0  0.0   93.0         6.5           1.0         0.0         1.0   

   season  hour  dayofweek  month  year  
0     3.0     0          6      1  2015  
1     3.0     1          6      1  2015  
2     3.0     2          6      1  2015  
3     3.0     3          6      1  2015  
4     3.0     4          6      1  2015  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17414 entries, 0 to 17413
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   cnt           17414 non-null  int64  
 1   t1            17414 non-

In [ ]:
target_column = 'cnt'
X_all = df.drop(target_column, axis=1).values.astype(np.float32)
y_all = df[target_column].values.astype(np.float32)

X = torch.from_numpy(X_all)
y = torch.from_numpy(y_all)

separator = DataSeparator(test_size=0.15, val_size=0.15, shuffle=True, seed=42)
data_splits = separator.separate(X, y)

scaler = StandardScaler()
scaler.fit(data_splits.X_train)

data_splits.X_train = scaler.transform(data_splits.X_train)
data_splits.X_val = scaler.transform(data_splits.X_val)
data_splits.X_test = scaler.transform(data_splits.X_test)

## Обучаем

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

results = {}

model1 = DecisionTree(classification=False, max_depth=6)
name = "Descision Tree"
print(f"Обучение модели: {name}")
model1.fit(data_splits.X_train, data_splits.y_train)
print("Обучение завершено.\n")

predictions = model1.predict(data_splits.X_test)

mse = mean_squared_error(data_splits.y_test.cpu().numpy(), predictions.cpu().numpy())
rmse = np.sqrt(mse)
mae = mean_absolute_error(data_splits.y_test.cpu().numpy(), predictions.cpu().numpy())

results[name] = {"MSE": mse, "RMSE": rmse, "MAE": mae}

Обучение модели: Descision Tree


/tmp/ipython-input-3088714251.py:16: UserWarning: var(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  return torch.var(y.float())


Обучение завершено.



In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

model2 = RandomForest(classification=False, max_depth=8, n_estimators=35)
name = "Random Forest"
print(f"Обучение модели: {name}")
model2.fit(data_splits.X_train, data_splits.y_train)
print("Обучение завершено.\n")

predictions = model2.predict(data_splits.X_test)

mse = mean_squared_error(data_splits.y_test.cpu().numpy(), predictions.cpu().numpy())
rmse = np.sqrt(mse)
mae = mean_absolute_error(data_splits.y_test.cpu().numpy(), predictions.cpu().numpy())

results[name] = {"MSE": mse, "RMSE": rmse, "MAE": mae}

Обучение модели: Random Forest
  Tree 1/35


/tmp/ipython-input-38744591.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float32)
/tmp/ipython-input-38744591.py:30: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(y, dtype=torch.int64 if self.classification else torch.float32)
/tmp/ipython-input-3088714251.py:16: UserWarning: var(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  return torch.var(y.float())


  Tree 2/35
  Tree 3/35
  Tree 4/35
  Tree 5/35
  Tree 6/35
  Tree 7/35
  Tree 8/35
  Tree 9/35
  Tree 10/35
  Tree 11/35
  Tree 12/35
  Tree 13/35
  Tree 14/35
  Tree 15/35
  Tree 16/35
  Tree 17/35
  Tree 18/35
  Tree 19/35
  Tree 20/35
  Tree 21/35
  Tree 22/35
  Tree 23/35
  Tree 24/35
  Tree 25/35
  Tree 26/35
  Tree 27/35
  Tree 28/35
  Tree 29/35
  Tree 30/35
  Tree 31/35
  Tree 32/35
  Tree 33/35
  Tree 34/35
  Tree 35/35
Обучение завершено.



/tmp/ipython-input-38744591.py:47: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float32)


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

model3 = GradientBoosting(classification=False, n_estimators=100, learning_rate=0.1, max_depth=3)
name = "Gradient Boosting"
print(f"Обучение модели: {name}")
model3.fit(data_splits.X_train, data_splits.y_train)
print("Обучение завершено.\n")

predictions = model3.predict(data_splits.X_test)

mse = mean_squared_error(data_splits.y_test.cpu().numpy(), predictions.cpu().numpy())
rmse = np.sqrt(mse)
mae = mean_absolute_error(data_splits.y_test.cpu().numpy(), predictions.cpu().numpy())

results[name] = {"MSE": mse, "RMSE": rmse, "MAE": mae}

Обучение модели: Gradient Boosting
  Tree 1/100


/tmp/ipython-input-3088714251.py:16: UserWarning: var(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  return torch.var(y.float())


  Tree 2/100
  Tree 3/100
  Tree 4/100
  Tree 5/100
  Tree 6/100
  Tree 7/100
  Tree 8/100
  Tree 9/100
  Tree 10/100
  Tree 11/100
  Tree 12/100
  Tree 13/100
  Tree 14/100
  Tree 15/100
  Tree 16/100
  Tree 17/100
  Tree 18/100
  Tree 19/100
  Tree 20/100
  Tree 21/100
  Tree 22/100
  Tree 23/100
  Tree 24/100
  Tree 25/100
  Tree 26/100
  Tree 27/100
  Tree 28/100
  Tree 29/100
  Tree 30/100
  Tree 31/100
  Tree 32/100
  Tree 33/100
  Tree 34/100
  Tree 35/100
  Tree 36/100
  Tree 37/100
  Tree 38/100
  Tree 39/100
  Tree 40/100
  Tree 41/100
  Tree 42/100
  Tree 43/100
  Tree 44/100
  Tree 45/100
  Tree 46/100
  Tree 47/100
  Tree 48/100
  Tree 49/100
  Tree 50/100
  Tree 51/100
  Tree 52/100
  Tree 53/100
  Tree 54/100
  Tree 55/100
  Tree 56/100
  Tree 57/100
  Tree 58/100
  Tree 59/100
  Tree 60/100
  Tree 61/100
  Tree 62/100
  Tree 63/100
  Tree 64/100
  Tree 65/100
  Tree 66/100
  Tree 67/100
  Tree 68/100
  Tree 69/100
  Tree 70/100
  Tree 71/100
  Tree 72/100
  Tree 73/100


In [ ]:
results_df = pd.DataFrame(results).T
print("--- Итоговое сравнение моделей ---")
print(results_df.sort_values(by='RMSE'))

--- Итоговое сравнение моделей ---
                             MSE        RMSE         MAE
Gradient Boosting  145540.015625  381.497072  250.536041
Descision Tree     253816.453125  503.801998  307.409485
Random Forest      370120.468750  608.375270  438.072754


### Топ моделей, топ 3 модели по RMSE

1. Random Forest
2. Gradient Boosting
3. Decision Tree

Почему топ-1 Random Forest. Случайный лес усредняет результаты большого количества деревьев, из-за чего снижается переобучение. А еще он менее чувствителен к выбросам, потому что у каждого дерева свой рандомный набор данных.

Gradient Boosting топ-2, **ПЕРЕПИСАТЬ**

Decision Tree сильно переобучается, чувствительно к выборсам, поэтому, ожидаемо, что оно хуже ансамблей.

# Решаем задачу классификации

## Работаем с датасетом

In [ ]:
import pandas as pd
import kagglehub
import tarfile
import pickle
import numpy as np
import os

path = kagglehub.dataset_download("pankrzysiu/cifar10-python")
file_name = "cifar-10-python.tar.gz"
full_path = f"{path}/{file_name}"

extract_dir = "/content/cifar10"
os.makedirs(extract_dir, exist_ok=True)

with tarfile.open(full_path, "r:gz") as tar:
    tar.extractall(path=extract_dir)

X_all = []
y_all = []

for i in range(1, 6):
    with open(f"{extract_dir}/cifar-10-batches-py/data_batch_{i}", "rb") as f:
        batch = pickle.load(f, encoding="bytes")
    X_all.append(batch[b"data"])
    y_all.extend(batch[b"labels"])

X = torch.from_numpy(np.vstack(X_all).astype(np.float32)[:10000])
y = torch.from_numpy(np.array(y_all).astype(np.int64)[:10000])

print(X.shape, y.shape)

print("\nПервые 5 записей X:")
print(X[:5])
print("\nПервые 5 записей y:")
print(y[:5])

Using Colab cache for faster access to the 'cifar10-python' dataset.


/tmp/ipython-input-873435418.py:16: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_dir)


torch.Size([10000, 3072]) torch.Size([10000])

Первые 5 записей X:
tensor([[ 59.,  43.,  50.,  ..., 140.,  84.,  72.],
        [154., 126., 105.,  ..., 139., 142., 144.],
        [255., 253., 253.,  ...,  83.,  83.,  84.],
        [ 28.,  37.,  38.,  ...,  28.,  37.,  46.],
        [170., 168., 177.,  ...,  82.,  78.,  80.]])

Первые 5 записей y:
tensor([6, 9, 9, 4, 1])


In [ ]:
separator = DataSeparator(test_size=0.15, val_size=0.15, shuffle=True, seed=42)
data_splits = separator.separate(X, y)

scaler = StandardScaler()
scaler.fit(data_splits.X_train)

data_splits.X_train = scaler.transform(data_splits.X_train)
data_splits.X_val = scaler.transform(data_splits.X_val)
data_splits.X_test = scaler.transform(data_splits.X_test)

## Обучаем

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score

results = {}

model1 = DecisionTree(classification=True, max_depth=6)
name = "Descision Tree"
print(f"Обучение модели: {name}")
model1.fit(data_splits.X_train, data_splits.y_train)
print("Обучение завершено.\n")

predictions = model1.predict(data_splits.X_test)

acc = accuracy_score(data_splits.y_test.cpu().numpy(), predictions)
f1 = f1_score(data_splits.y_test.cpu().numpy(), predictions, average="macro")
precision = precision_score(data_splits.y_test.cpu().numpy(), predictions, average="macro")

results[name] = {"accuracy": acc, "f1": f1, "precision": precision}

Обучение модели: Descision Tree
Обучение завершено.



In [ ]:
results_df = pd.DataFrame(results).T
print("--- Итоговое сравнение моделей ---")
print(results_df.sort_values(by='accuracy'))

--- Итоговое сравнение моделей ---
                accuracy        f1  precision
Descision Tree  0.251333  0.239249   0.239063


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score

model2 = RandomForest(classification=True, max_depth=8, n_estimators=30)
name = "Random Forest"
print(f"Обучение модели: {name}")
model2.fit(data_splits.X_train, data_splits.y_train)
print("Обучение завершено.\n")

predictions = model2.predict(data_splits.X_test)

acc = accuracy_score(data_splits.y_test.cpu().numpy(), predictions)
f1 = f1_score(data_splits.y_test.cpu().numpy(), predictions, average="macro")
precision = precision_score(data_splits.y_test.cpu().numpy(), predictions, average="macro")

results[name] = {"accuracy": acc, "f1": f1, "precision": precision}

Обучение модели: Random Forest
  Tree 1/30


/tmp/ipython-input-38744591.py:29: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float32)
/tmp/ipython-input-38744591.py:30: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  y = torch.tensor(y, dtype=torch.int64 if self.classification else torch.float32)


  Tree 2/30
  Tree 3/30
  Tree 4/30
  Tree 5/30
  Tree 6/30
  Tree 7/30
  Tree 8/30
  Tree 9/30
  Tree 10/30
  Tree 11/30
  Tree 12/30
  Tree 13/30
  Tree 14/30
  Tree 15/30
  Tree 16/30
  Tree 17/30
  Tree 18/30
  Tree 19/30
  Tree 20/30
  Tree 21/30
  Tree 22/30
  Tree 23/30
  Tree 24/30
  Tree 25/30
  Tree 26/30
  Tree 27/30
  Tree 28/30
  Tree 29/30
  Tree 30/30
Обучение завершено.



/tmp/ipython-input-38744591.py:47: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  X = torch.tensor(X, dtype=torch.float32)


In [ ]:
results_df = pd.DataFrame(results).T
print("--- Итоговое сравнение моделей ---")
print(results_df.sort_values(by='accuracy'))

--- Итоговое сравнение моделей ---
                accuracy        f1  precision
Descision Tree  0.251333  0.239249   0.239063
Random Forest   0.357333  0.346768   0.371236


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, precision_score

model3 = GradientBoosting(classification=True, n_estimators=50, learning_rate=0.05, max_depth=3)
name = "Gradient Boosting"
print(f"Обучение модели: {name}")
model3.fit(data_splits.X_train, data_splits.y_train)
print("Обучение завершено.\n")

predictions = model3.predict(data_splits.X_test)

acc = accuracy_score(data_splits.y_test.cpu().numpy(), predictions)
f1 = f1_score(data_splits.y_test.cpu().numpy(), predictions, average="macro")
precision = precision_score(data_splits.y_test.cpu().numpy(), predictions, average="macro")

results[name] = {"accuracy": acc, "f1": f1, "precision": precision}

Обучение модели: Gradient Boosting
  Estimator 1/50


/tmp/ipython-input-3088714251.py:16: UserWarning: var(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1857.)
  return torch.var(y.float())


  Estimator 2/50
  Estimator 3/50
  Estimator 4/50
  Estimator 5/50
  Estimator 6/50
  Estimator 7/50
  Estimator 8/50
  Estimator 9/50
  Estimator 10/50
  Estimator 11/50
  Estimator 12/50
  Estimator 13/50
  Estimator 14/50
  Estimator 15/50
  Estimator 16/50


In [ ]:
results_df = pd.DataFrame(results).T
print("--- Итоговое сравнение моделей ---")
print(results_df.sort_values(by='accuracy'))